# 엔진 검증 — 몬테카를로 (데이터 불필요)

귀무가설(α 전부 0) + 정규 잔차에서 합성 데이터를 수천 번 만들면, GRS 통계량은
유한표본에서 정확히 $F(N,\,T-N-K)$를 따라야 한다. 5% 기각률이 ~5%로 나오고
분포가 일치하면 **수식·자유도가 동시에 검증**된다.

> `tests/` 폴더에서 실행 — **실제 엔진(`ff_core.py`)의 `grs_test`를 직접 불러와** 검증한다(복사본이 아니라 진짜 코드).

In [1]:
import sys; sys.path.insert(0, '../src')    # src/ff_core.py 불러오기
import numpy as np
from scipy import stats
from ff_core import grs_test                     # 실제 엔진의 GRS
RNG = np.random.default_rng(20260628)

def fast_alpha_resid(Y, F):
    X = np.column_stack([np.ones(len(Y)), F])
    coef = np.linalg.solve(X.T @ X, X.T @ Y)
    return coef[0], Y - X @ coef

def simulate(T, N, K, reps, alpha_inject=0.0):
    rej = 0; Fs = np.empty(reps)
    scale = np.array([1.0] + [0.6]*(K-1)); mean = np.array([0.5] + [0.25]*(K-1))
    for r in range(reps):
        F = RNG.normal(size=(T, K)) * scale + mean
        B = RNG.uniform(0.5, 1.5, size=(N, K)); E = RNG.normal(size=(T, N))
        Y = np.full(N, alpha_inject) + F @ B.T + E
        a, res = fast_alpha_resid(Y, F)
        out = grs_test(a, res, F)                 # 엔진 함수 호출
        Fs[r] = out['F']
        if out['p_value'] < 0.05: rej += 1
    return rej / reps, Fs

## 크기 검정 — 귀무가설하 기각률이 명목수준과 맞는가

In [2]:
T, N, K, reps = 720, 25, 3, 5000
size, Fs = simulate(T, N, K, reps, 0.0)
ks = stats.kstest(Fs, lambda x: stats.f.cdf(x, N, T - N - K))
print(f"5% 기각률 : {size:.4f}   (목표 0.0500)")
print(f"통계량 평균: {Fs.mean():.4f}   (이론 {(T-N-K)/(T-N-K-2):.4f})")
print(f"KS vs F분포: p = {ks.pvalue:.4f}   (크면 분포 일치)")
print('판정:', '엔진 검증 통과 — 수식·자유도 정확'
      if abs(size - 0.05) < 0.01 and ks.pvalue > 0.05 else '재확인 필요')

5% 기각률 : 0.0500   (목표 0.0500)
통계량 평균: 1.0010   (이론 1.0029)
KS vs F분포: p = 0.8707   (크면 분포 일치)
판정: 엔진 검증 통과 — 수식·자유도 정확


## 검정력 — α를 주입하면 기각률이 올라가는가

In [3]:
for a in (0.05, 0.10):
    p5, _ = simulate(T, N, K, 1500, a)
    print(f"alpha = {a:.2f}  →  5% 기각률 {p5:.4f}")

alpha = 0.05  →  5% 기각률 0.8913


alpha = 0.10  →  5% 기각률 1.0000
